# Count change: does losing more seizure types go with a better outcome?

`disappearance.ipynb` (this folder) looks at one seizure type at a time. This looks at the whole patient: how many seizure types they had before surgery, how many after, and how many they lost.

**Partly circular.** Engel and ILAE classes measure seizure reduction, so a patient who stops having one or two seizure types is likely to land in a better class almost by definition; and patients with more types before surgery have more to lose. So these results describe what changed after surgery, not evidence that losing types explains who improved.

Runs on one **unit** (`UNIT`, the first code cell; see `cc_stats/units.py`). The count is over the unit's `count_groups`: for `types` the five core groups (atonic, absence, BTC, tonic, myoclonic), so each count runs from 0 to 5; for `codes` the 12 codes with at least 10 patients before surgery, focal awareness codes aside (a focal seizure counts once, through its feature), so 0 to 12. Cohort: patients with every counted group recorded before and after surgery, and the outcome recorded. The types notebook is the source; the codes one is generated from it by `analysis/unit_notebooks.py` (only `UNIT` and the notes differ), so edit the types notebook and regenerate. Below, "type" means a counted group of the unit.

Every table and figure is shown for both outcomes, Engel then ILAE (see `cc_stats/outcomes.py`); the scripts compute both.

- **Net loss:** types before minus types after. Negative means the patient has more types recorded after surgery than before.
- **Change group:** gained (net loss below 0), no change (0), lost 1, lost 2+.

`count_change.py` computes the counts, the Spearman correlations and the improvement rates. `count_change_firth.R` fits the Firth models. This notebook runs both; the outputs go to `results/<version>/2_disappearance/<unit>/`.

In [ ]:
# The unit this notebook runs on: 'types' or 'codes' (cc_stats/units.py).
# Set here only; the codes notebook is generated with 'codes' (analysis/unit_notebooks.py).
UNIT = 'types'

In [ ]:
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from cc_stats.outcomes import OUTCOMES, heading
from cc_stats.paths import DATASET, PROJECT_ROOT, results_dir
from cc_stats.plots import PAIR_A, rate_plot, save_figure
from cc_stats.rscript import find_rscript

print(f'dataset {DATASET}')
run = subprocess.run([sys.executable, 'analysis/2_disappearance/count_change.py', UNIT],
                     capture_output=True, text=True, cwd=PROJECT_ROOT)
if run.returncode != 0:
    raise RuntimeError(f'count_change.py failed:\n{run.stderr}')
print('\n'.join(l for l in run.stdout.splitlines() if l.split(':')[0] in OUTCOMES))
run = subprocess.run([find_rscript(), 'analysis/2_disappearance/count_change_firth.R', UNIT],
                     capture_output=True, text=True, cwd=PROJECT_ROOT)
if run.returncode != 0:
    raise RuntimeError(f'count_change_firth.R failed:\n{run.stderr}')

# The scripts compute both outcomes; split each result by outcome.
FOLDER = f'2_disappearance/{UNIT}'
RESULTS = results_dir(FOLDER)
COUNTED = 'core seizure types' if UNIT == 'types' else 'seizure codes'
# 'type' or 'code', for figure titles and labels
WORD = 'type' if UNIT == 'types' else 'code'
def by_outcome(f, **kw):
    table = pd.read_csv(RESULTS / f, **kw)
    split = {outcome: table.query('outcome == @outcome') for outcome in OUTCOMES}
    # Keep an index read from the file (subject_id); renumber the rest from 0
    return split if 'index_col' in kw else {o: t.reset_index(drop=True) for o, t in split.items()}
counts = by_outcome('count_change_by_patient.csv', index_col='subject_id')
correlations = by_outcome('count_correlations.csv')
by_change = by_outcome('improvement_by_count_change.csv')
firth = by_outcome('count_change_firth.csv')


def fmt_p(v):
    return '<0.001' if v < 0.001 else f'{v:.3f}'

## How the counts change

Rows: types before surgery. Columns: types after. The counts don't depend on the outcome, but each cohort needs its outcome recorded, so the two tables differ slightly.

In [ ]:
for outcome in OUTCOMES:
    heading(outcome)
    c = counts[outcome]
    display(pd.crosstab(c['n_before'], c['n_after'], margins=True, rownames=['before'], colnames=['after']))
    print(f"{(c['n_gained'] > 0).sum()} patients gained at least one type after surgery "
          f"(some of them also lost another)")

## Correlation with the outcome class

Spearman's rho between each count and the outcome class (Engel I-IV or ILAE 1-6). **Higher class is worse**, so a negative rho means *more of this goes with a better outcome*. The 95% CI uses the Fisher z transform with the Bonett-Wright standard error.

In [ ]:
for outcome, O in OUTCOMES.items():
    heading(outcome)
    r = correlations[outcome].copy()
    # The scripts label the counts 'types before' / 'types after' for either unit
    r['label'] = r['label'].str.replace('types', f'{WORD}s')
    display(pd.DataFrame({
        'Count': r['label'],
        'n': r['n'],
        'rho (95% CI)': [f'{v:+.2f} ({lo:+.2f} to {hi:+.2f})' for v, lo, hi in r[['rho', 'lo', 'hi']].to_numpy()],
        'p': r['p'].map(fmt_p),
    }).style.hide(axis='index'))

    fig, ax = plt.subplots(figsize=(9, 2.8))
    y = np.arange(len(r))[::-1]
    ax.errorbar(r['rho'], y, xerr=np.clip([r['rho'] - r['lo'], r['hi'] - r['rho']], 0, None), fmt='o', ms=7,
                color=PAIR_A, elinewidth=2, capsize=4, capthick=2)
    ax.axvline(0, color='#888888', lw=1)
    ax.set_yticks(y, r['label'])
    ax.set_ylim(-0.6, len(r) - 0.4)
    ax.set_xlim(-1, 1)
    ax.set_xlabel(f"Spearman rho with {O['name']} class (negative = better outcome)")
    ax.set_title(f"{O['name']}: {WORD} counts vs outcome class (n = {r['n'].iloc[0]})", loc='left', pad=20)
    # rho and p beside each row, in headed columns
    text = dict(transform=ax.get_yaxis_transform(), va='center', ha='center', fontsize=9)
    for x, header in ((1.16, 'rho (95% CI)'), (1.40, 'p')):
        ax.text(x, len(r) - 0.35, header, fontweight='bold', **text)
    for yi, row in zip(y, r.itertuples()):
        ax.text(1.16, yi, f'{row.rho:+.2f} ({row.lo:+.2f} to {row.hi:+.2f})', **text)
        ax.text(1.40, yi, fmt_p(row.p), **text)
    ax.grid(axis='x', color='#e5e5e5')
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    plt.tight_layout()
    save_figure(FOLDER, f'count_correlations_{outcome}')
    plt.show()

## Improvement by change in type count

Share of patients improved in each change group, with Wilson 95% CIs.

In [ ]:
for outcome, O in OUTCOMES.items():
    heading(outcome)
    b = by_change[outcome]
    display(pd.DataFrame({
        'Change': b['change'],
        'Improved / n': b['improved'].astype(str) + '/' + b['n'].astype(str),
        'Rate (95% CI)': [f'{r:.0%} ({lo:.0%}-{hi:.0%})' for r, lo, hi in b[['rate', 'lo', 'hi']].to_numpy()],
    }).style.hide(axis='index'))

    ax = rate_plot(b['change'], b['rate'], b['lo'], b['hi'], figsize=(8, 3.2),
                   counts=(b['improved'], b['n']), header='Improved / total')
    ax.set_xlabel(f"Improved ({O['improved_label']})")
    ax.set_title(f"{O['name']}: improvement by change in the number of {COUNTED}", loc='left', pad=20)
    plt.tight_layout()
    save_figure(FOLDER, f'improvement_by_count_change_{outcome}')
    plt.show()

## Firth models

One predictor per model: `improved ~ count`. The OR is per one more type (for net loss: per one more type lost). OR above 1 means higher odds of improvement.

In [ ]:
for outcome in OUTCOMES:
    heading(outcome)
    f = firth[outcome]
    display(pd.DataFrame({
        'Predictor': f['predictor'],
        'n / not improved': f['n'].astype(str) + ' / ' + f['not_improved'].astype(str),
        'OR (95% CI)': [f'{o:.2f} ({lo:.2f}-{hi:.2f})' for o, lo, hi in f[['OR', 'lo', 'hi']].to_numpy()],
        'p': f['p'].map(fmt_p),
    }).style.hide(axis='index'))

## Reading the results

- **Partly circular.** Engel and ILAE classes measure seizure reduction, so a patient who stops having one or two seizure types is likely to land in a better class almost by definition; and patients with more types before surgery have more to lose. So these results describe what changed after surgery, not evidence that losing types explains who improved.
- A single correlation or OR assumes a steady trend: each extra type lost shifts the outcome the same way. The improvement-by-change table shows whether that is actually the shape: whether improvement rises steadily with types lost, or the groups at both ends behave alike.
- Groups with no not-improved patients (often "lost 2+", and on Engel also "gained") show 100%. With 16 (Engel) or 22 (ILAE) not-improved patients in all, read those as "no failures seen", with the wide CIs shown.
- Checked 2026-10-20 on v2: the two outcomes disagree here. On Engel there is no trend (net loss rho -0.10). On ILAE, losing more types goes with a better class (rho -0.26, -0.45 to -0.04, p = 0.019), and improvement rises from 65% (no change) to 90% (lost 1) and 100% (lost 2+). ILAE counts seizure days, so it may track the loss of seizure types more closely than Engel does.